# 🤖 Working with AI APIs — Google Gemini (FREE)
### AI Bootcamp · Day 3 · DigiPen Institute of Technology

> **How to use this notebook:**  
> Read each section. Run each code cell with **Shift + Enter**. Do NOT skip ahead.
> Every cell builds on the previous one.

| Topic | What you learn |
|---|---|
| 📡 Topic 1 | What is an API? JSON basics |
| 🧠 Topic 2 | Tokens, messages, temperature |
| 🔑 Topic 3 | Getting your FREE Gemini API key |
| 🔌 Topic 4 | Your first real AI call |
| 💻 Topic 5 | Safe function + streaming |
| 🚀 Topic 6 | Build a personality bot + memory bot |

**Cost for today:** $0.00 — Gemini free tier, no credit card needed.  
**Requirement:** Google account (any Gmail works).

## ⚠️ Install the correct library first
Run this in your terminal (inside your venv):
```
pip uninstall google-generativeai -y
pip install google-genai python-dotenv tiktoken
```
The old library (`google-generativeai`) is deprecated and will cause errors.
The new library is `google-genai`. They are different — use only the new one.


---
# 📡 Topic 1 — What is an API?

**API** stands for **A**pplication **P**rogramming **I**nterface.

Think of it like a restaurant:
- **YOU** (your Python code) = the customer
- **MENU** (API documentation) = the list of things you can ask for
- **WAITER** (the API) = takes your request, brings back the result
- **KITCHEN** (Google's server) = does the actual AI work — you never see inside

**Key insight:** The Gemini web chat is just an API with a nice website on top.  
Today you will call that SAME API directly from Python — no website needed.

### ⚠️ Important: LLMs can HALLUCINATE
LLMs sometimes generate text that sounds confident but is factually wrong.  
Always verify important facts. Treat the AI as a brilliant but occasionally unreliable assistant.


## 1.1 JSON — The Language APIs Speak

Every piece of data going TO an API, and every response coming BACK, is in **JSON** format.

JSON looks almost exactly like a Python dictionary. Key differences:
- Always **double quotes** for keys: `"name"` not `'name'`
- Lowercase: `true`, `false`, `null` — NOT Python's `True`, `False`, `None`
- No trailing comma after the last item

**Two functions you must know:**
- `json.dumps()` = Python dict → JSON string (for **SENDING** to an API)
- `json.loads()` = JSON string → Python dict (for **RECEIVING** from an API)


In [1]:
# Cell 1.1: JSON basics — run this cell with Shift+Enter
import json   # built-in Python library — no pip install needed

student = {
    "name":    "Aisha",
    "year":    2,
    "gpa":     3.85,
    "active":  True,   # Python True becomes JSON true
    "modules": ["CS2030", "CS2040"],
    "advisor": None    # Python None becomes JSON null
}

# json.dumps() = convert Python dict → JSON string (for SENDING)
json_string = json.dumps(student, indent=2)
print("=== STEP 1: What we SEND to the API ===")
print(json_string)
print(f'Type: {type(json_string)}  ← STRING, not dict!')
print()

# json.loads() = convert JSON string → Python dict (for RECEIVING)
api_response = '{"answer":"Python is great!","tokens":23}'
parsed = json.loads(api_response)
print("=== STEP 2: What we RECEIVE from the API ===")
print(f'Answer: {parsed["answer"]}')
print(f'Tokens: {parsed["tokens"]}')
print(f'Type: {type(parsed)}  ← DICT!')
print()

# Round-trip verification: does everything survive JSON conversion?
received_back = json.loads(json_string)
print("=== STEP 3: Round-Trip Verification ===")
fields = ["name", "year", "gpa", "active", "modules", "advisor"]
for field in fields:
    original = student[field]
    restored = received_back[field]
    status = "OK" if original == restored else "MISMATCH"
    print(f'  {field:<10}: {status}  | {original!r}')
print("\nAll fields survived! JSON is lossless.")


=== STEP 1: What we SEND to the API ===
{
  "name": "Aisha",
  "year": 2,
  "gpa": 3.85,
  "active": true,
  "modules": [
    "CS2030",
    "CS2040"
  ],
  "advisor": null
}
Type: <class 'str'>  ← STRING, not dict!

=== STEP 2: What we RECEIVE from the API ===
Answer: Python is great!
Tokens: 23
Type: <class 'dict'>  ← DICT!

=== STEP 3: Round-Trip Verification ===
  name      : OK  | 'Aisha'
  year      : OK  | 2
  gpa       : OK  | 3.85
  active    : OK  | True
  modules   : OK  | ['CS2030', 'CS2040']
  advisor   : OK  | None

All fields survived! JSON is lossless.


## 🧪 Self-Check — Topic 1
1. What does API stand for? Explain each word.
2. What is the difference between `json.dumps()` and `json.loads()`?
3. Fix this broken JSON: `{'name': 'James', 'active': True, 'score': None,}`
4. Name 3 apps you use that rely on APIs.


---
# 🧠 Topic 2 — How LLM APIs Work

Before writing real API code, you need to understand 3 concepts:
**Tokens**, **Messages**, and **Temperature**.


## 2.1 Tokens — The Currency of Everything

LLMs do **NOT** process words. They process **tokens** — sub-word chunks of text.

**Rule of thumb:** 1 token ≈ 0.75 English words, or 1 word ≈ 1.33 tokens

**Why 1.33?**  
Average English word = ~4.5 characters.  
GPT tokeniser encodes ~3.4 characters per token.  
So: tokens per word = 4.5 ÷ 3.4 = **1.32 ≈ 1.33**

> Note: We use the OpenAI tiktoken library for token estimation.
> Gemini uses its own internal tokeniser, but counts are similar for English text.


In [3]:
# Cell 2.1: Count tokens using tiktoken (works as good estimate for Gemini too)
# !pip install tiktoken   ← run this if you get ModuleNotFoundError
import tiktoken

# gpt-4o tokeniser — a good approximation for Gemini token counts
enc = tiktoken.encoding_for_model('gpt-4o')
print('Tokeniser loaded! (Approximate counts for Gemini)')

texts = [
    'hello',
    'running',
    'ChatGPT',
    'supercalifragilistic',
    'DigiPen',
    'What is machine learning?',
    'The AI bootcamp teaches DigiPen students to build AI apps.',
]

print(f"{'Text':<52} | Tokens | Words | Ratio")
print('-' * 75)
for text in texts:
    tok   = len(enc.encode(text))
    words = len(text.split())
    ratio = tok / words if words > 0 else 0
    print(f'{text:<52} | {tok:>6} | {words:>5} | {ratio:.2f}')

print()
print('Rule: token count ≈ word count × 1.33')


Tokeniser loaded! (Approximate counts for Gemini)
Text                                                 | Tokens | Words | Ratio
---------------------------------------------------------------------------
hello                                                |      1 |     1 | 1.00
running                                              |      1 |     1 | 1.00
ChatGPT                                              |      2 |     1 | 2.00
supercalifragilistic                                 |      6 |     1 | 6.00
DigiPen                                              |      3 |     1 | 3.00
What is machine learning?                            |      5 |     4 | 1.25
The AI bootcamp teaches DigiPen students to build AI apps. |     13 |    10 | 1.30

Rule: token count ≈ word count × 1.33


## 2.2 The Messages Array — How the AI Reads Your Request

LLM APIs take a **list of messages** rather than a plain text string.
Each message has a `role` and `content`.

| Role | Purpose |
|---|---|
| `system` | AI's rules and persona — applies to every reply |
| `user` | Your question or instruction |
| `assistant` | The AI's previous replies — used for conversation memory |

> **Gemini note:** In `client.chats.create()`, Gemini handles the history automatically.  
> You do NOT need to manually manage a history list — Gemini does it for you.

### ⚡ CRITICAL: LLMs are COMPLETELY STATELESS
Every single API call starts with **zero memory**.  
The messages you send IS the only memory.


In [4]:
# Cell 2.2: Demonstrate statelessness — no API key needed
def fake_api_call(messages):
    '''Simulates an AI — shows the statelessness concept.'''
    last = messages[-1]['content']
    if 'Priya' in last:
        return 'Nice to meet you, Priya! CS at DigiPen is a great choice!'
    elif 'name' in last.lower() and len(messages) == 1:
        return 'I do not know your name. You have not told me.'
    elif 'name' in last.lower() and len(messages) > 1:
        return 'Your name is Priya! You told me in your first message.'
    return 'I can help with that.'

# TURN 1
print('TURN 1 — User introduces themselves')
messages_turn1 = [
    {'role': 'user', 'content': 'Hi! My name is Priya and I study CS at DigiPen.'}
]
reply1 = fake_api_call(messages_turn1)
print(f'You: {messages_turn1[0]["content"]}')
print(f'Bot: {reply1}')

# TURN 2 WRONG — no history
print()
print('TURN 2 WRONG — New call, no history')
messages_bad = [{'role': 'user', 'content': 'What is my name?'}]
print(f'Bot: {fake_api_call(messages_bad)}')
print('The AI forgot Priya! Each call starts with zero memory.')

# TURN 2 CORRECT — include history
print()
print('TURN 2 CORRECT — Full history included')
messages_good = [
    {'role': 'user',      'content': 'Hi! My name is Priya and I study CS at DigiPen.'},
    {'role': 'assistant', 'content': reply1},
    {'role': 'user',      'content': 'What is my name?'}
]
print(f'Bot: {fake_api_call(messages_good)}')
print('The AI remembered! Because we included the history.')


TURN 1 — User introduces themselves
You: Hi! My name is Priya and I study CS at DigiPen.
Bot: Nice to meet you, Priya! CS at DigiPen is a great choice!

TURN 2 WRONG — New call, no history
Bot: I do not know your name. You have not told me.
The AI forgot Priya! Each call starts with zero memory.

TURN 2 CORRECT — Full history included
Bot: Your name is Priya! You told me in your first message.
The AI remembered! Because we included the history.


## 2.3 Temperature — The Creativity Dial

| Value | Behaviour | Use for |
|---|---|---|
| `0.0` | Same answer every time | Maths, code, facts |
| `0.7` | Balanced — **start here** | General chat, explanations |
| `1.0` | More creative | Brainstorming, storytelling |
| `1.5+` | Very creative, may be inconsistent | Comedy, poetry |


## 🧪 Self-Check — Topic 2
1. How many tokens is a 500-word essay approximately?
2. What are the 3 message roles? What does each one do?
3. You ask the AI your name and it says it does not know. What went wrong?
4. Should a maths quiz checker use temperature 0.0 or 1.5? Why?


---
# 🔑 Topic 3 — Getting Your FREE Gemini API Key

## 3.1 Sign Up for Google AI Studio
1. Go to **aistudio.google.com**
2. Sign in with any **Google account** (personal Gmail is fine)
3. Click **Get API key** in the left sidebar
4. Click **Create API key** → select **Create API key in new project**
5. **COPY IT IMMEDIATELY** — it starts with `AIzaSy...`

## 3.2 No Spending Limit Needed
> ✅ Gemini free tier is completely free. No credit card. No spending limit to set.

**Free tier limits:** 15 requests per minute, 1 million tokens per minute.  
For a classroom session this is plenty — just add `time.sleep(2)` between calls in loops.

## 3.3 Which Model to Use
Run this cell to see which models are available on your account:


In [31]:
# Cell 3.0: Check which Gemini models are available on your account
# Run this FIRST to find a model that works for you
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))

print('Available models on your account:')
print('-' * 50)
for m in client.models.list():
    # Only show generative models (not embedding models)
    if 'generateContent' in str(getattr(m, 'supported_actions', '')):
        print(m.name)

print()
print('Use one of the models listed above.')
print('Try: gemini-2.0-flash or gemini-1.5-flash')
print('If one gives a quota error, try another from the list.')


Available models on your account:
--------------------------------------------------
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-

## 3.4 Set the Model Name

> **Run the cell above first to see your available models.**  
> Then set your model name in the cell below.  
> Common options: `gemini-2.0-flash`, `gemini-1.5-flash`, `gemini-1.5-flash-latest`

## 3.5 Create Your .env File
Create a file called `.env` in the **same folder as this notebook**.  
Open it in Notepad (NOT Jupyter) and add:
```
GOOGLE_API_KEY=AIzaSy-your-actual-key-here
```
Rules: no spaces around `=`, no quotes, no `.txt` extension.

## 3.6 Create Your .gitignore
```
.env
venv/
__pycache__/
```


In [17]:
# Cell 3.1: Set your model name — change this if you get a quota error
# Run Cell 3.0 above first to see which models are available to you

# ── SET YOUR MODEL NAME HERE ─────────────────────────────────────────
GEMINI_MODEL = 'gemini-3.1-flash-lite'   # change this if you get a 429 error
# ─────────────────────────────────────────────────────────────────────

print(f'Model set to: {GEMINI_MODEL}')
print('If you get a quota error (429), change GEMINI_MODEL above.')
print('Try: gemini-1.5-flash or gemini-1.5-flash-latest')


Model set to: gemini-3.1-flash-lite
If you get a quota error (429), change GEMINI_MODEL above.
Try: gemini-1.5-flash or gemini-1.5-flash-latest


In [9]:
# Cell 3.2: Verify your API key is working
from google import genai          # the NEW official Gemini library
from google.genai import types    # for GenerateContentConfig
from dotenv import load_dotenv
import os

# load_dotenv() reads .env and puts GOOGLE_API_KEY into the environment
# MUST run before creating the client
load_dotenv()

key = os.getenv('GOOGLE_API_KEY')
if not key:
    print('Key NOT found. Check:')
    print('  1. Does .env exist in the SAME folder as this notebook?')
    print('  2. Is it named .env (not .env.txt)?')
    print('  3. Is GOOGLE_API_KEY spelled correctly?')
    print()
    print('Files in this folder:', os.listdir('.'))
else:
    print(f'Key found! First 8 chars: {key[:8]}...')

    # Create the Gemini client
    client = genai.Client(api_key=key)

    # Quick test call
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents='Say hello in exactly 3 words.',
    )
    print(f'Gemini says: {response.text}')
    print('Setup complete! You are ready for Topic 4.')


Key found! First 8 chars: AQ.Ab8RN...
Gemini says: Hello there, friend.
Setup complete! You are ready for Topic 4.


In [10]:
from google import genai

client = genai.Client(api_key=key)

print(client)

---
# 🔌 Topic 4 — Your First Real API Call

This is the moment. We call a real AI from Python.

## How Gemini differs from OpenAI

| Feature | OpenAI | Gemini |
|---|---|---|
| Library | `from openai import OpenAI` | `from google import genai` |
| Client | `OpenAI()` | `genai.Client(api_key=...)` |
| Call method | `.chat.completions.create()` | `.models.generate_content()` |
| System prompt | Inside `messages` list | `system_instruction=` in config |
| Get reply | `.choices[0].message.content` | `.text` |
| Max tokens | `max_tokens=` | `max_output_tokens=` in config |
| Streaming | `.chat.completions.create(stream=True)` | `.models.generate_content_stream()` |
| Memory | Manual history list | Built-in: `client.chats.create()` |


In [18]:
# Cell 4.1: Setup — run this first in EVERY notebook

# ── IMPORTANT: use 'from google import genai' (NOT 'import google.generativeai') ──
# 'import google.generativeai' loads the OLD deprecated library → AttributeError
# 'from google import genai'   loads the NEW correct library → works
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv()   # read .env — must run before creating client

# Create the Gemini client
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))

print('Client ready! We are connected to Gemini.')
print(f'Using model: {GEMINI_MODEL}')
print('(Change GEMINI_MODEL in Topic 3 Cell 3.1 if you get a 429 error)')


Client ready! We are connected to Gemini.
Using model: gemini-3.1-flash-lite
(Change GEMINI_MODEL in Topic 3 Cell 3.1 if you get a 429 error)


In [19]:
# Cell 4.2: Your first real Gemini API call — every parameter explained

response = client.models.generate_content(

    # Which model to use — set in Topic 3 Cell 3.1
    model=GEMINI_MODEL,

    # Your question — a plain string for a single question
    contents='What is Python programming language?',

    # All settings go inside GenerateContentConfig
    config=types.GenerateContentConfig(

        # system_instruction = the AI's rules (like 'system' role in OpenAI)
        # Applies to EVERY reply in this session
        system_instruction='You are a friendly assistant. Answer in exactly 2 sentences.',

        # Creativity dial — 0.7 is balanced, use this as your default
        temperature=0.7,

        # Maximum tokens in the response — ALWAYS set this
        max_output_tokens=100,
    )
)

# Get the AI's reply — response.text is the plain string
answer = response.text
print('AI says:')
print(answer)
print()

# Check finish reason
# FinishReason.STOP     = complete response ✅
# FinishReason.MAX_TOKENS = cut off — increase max_output_tokens
finish = response.candidates[0].finish_reason
print(f'Finish reason: {finish}')

# Token usage
usage = response.usage_metadata
print(f'Tokens: {usage.prompt_token_count} in + {usage.candidates_token_count} out')
print('Cost: $0.00  (Gemini free tier!)')


AI says:
Python is a high-level, interpreted programming language known for its clear syntax and readability. It is widely used in fields like web development, data science, and automation because of its versatility and vast library support.

Finish reason: FinishReason.STOP
Tokens: 21 in + 42 out
Cost: $0.00  (Gemini free tier!)


## 4.3 Understanding the Gemini Response Object

| Path | What it gives you |
|---|---|
| `response.text` | The AI's reply as a plain string |
| `response.candidates[0].finish_reason` | `FinishReason.STOP` = complete, `FinishReason.MAX_TOKENS` = cut off |
| `response.usage_metadata.prompt_token_count` | Tokens YOU sent |
| `response.usage_metadata.candidates_token_count` | Tokens AI replied with |

> ⚠️ **`FinishReason.MAX_TOKENS` is a silent issue.** No error is raised.  
> The response is just cut off. Always check finish_reason after every call.


In [20]:
# Cell 4.3: Temperature experiment
import time

def temp_test(temperature, prompt, runs=5):
    print(f'temperature = {temperature}')
    print('-' * 45)
    results = []
    for i in range(1, runs + 1):
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temperature,
                max_output_tokens=10,
            )
        )
        answer = response.text.strip()
        results.append(answer)
        print(f'  Run {i}: {answer}')
        time.sleep(3)   # important: free tier = 15 requests/minute
    if len(set(results)) == 1:
        print('  → Runs IDENTICAL (expected at low temperature)')
    else:
        print('  → Runs DIFFERENT (expected at high temperature)')
    print()

prompt = 'Give me exactly ONE word meaning happy. Only the word.'
temp_test(0.0, prompt)   # always the same
time.sleep(5)            # pause between groups
temp_test(1.5, prompt)   # often different


temperature = 0.0
---------------------------------------------
  Run 1: Joyful
  Run 2: Joyful
  Run 3: Joyful
  Run 4: Joyful
  Run 5: Joyful
  → Runs IDENTICAL (expected at low temperature)

temperature = 1.5
---------------------------------------------
  Run 1: Joyful
  Run 2: Joyful
  Run 3: Joyful
  Run 4: Joyful
  Run 5: Joyful
  → Runs IDENTICAL (expected at low temperature)



## 🧪 Self-Check — Topic 4
1. What is `response.text` in Gemini? What is the equivalent in OpenAI?
2. What does `FinishReason.MAX_TOKENS` mean? What should you do?
3. Where does the system instruction go in a Gemini call?
4. Why must `load_dotenv()` run before `genai.Client()`?


---
# 💻 Topic 5 — Full Working Example

The raw API call works, but has problems:
- What if you hit the rate limit? → it crashes
- What if the response is cut off? → no warning

The `ask()` function below handles all of this.

> ⚠️ **Free tier rate limit:** 15 requests per minute.  
> Always add `time.sleep(2)` between calls when running in a loop.


In [21]:
# Cell 5.1: Safe ask() function for Gemini
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os, time

load_dotenv()
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))

def ask(prompt, system='You are a helpful assistant.', temperature=0.7, max_tokens=200):
    '''
    Safe Gemini call with auto-retry on rate limit.
    Returns: dict with answer, finish, tokens_in, tokens_out
    Returns: None if something fails permanently
    '''
    # Retry up to 3 times — handles temporary rate limits
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    temperature=temperature,
                    max_output_tokens=max_tokens,
                )
            )

            text   = response.text
            finish = response.candidates[0].finish_reason
            in_t   = response.usage_metadata.prompt_token_count
            out_t  = response.usage_metadata.candidates_token_count

            if 'MAX_TOKENS' in str(finish):
                print('⚠️  Response cut off — increase max_tokens')

            return {'answer': text, 'finish': finish,
                    'tokens_in': in_t, 'tokens_out': out_t}

        except Exception as e:
            err = str(e)
            if '429' in err or 'quota' in err.lower() or 'RESOURCE_EXHAUSTED' in err:
                # Rate limit — wait and retry (exponential backoff: 1s then 2s)
                if attempt < 2:
                    wait = 2 ** attempt
                    print(f'⏳ Rate limited. Waiting {wait}s... (attempt {attempt+1}/3)')
                    time.sleep(wait)
                else:
                    print('❌ Rate limit — failed after 3 attempts. Wait 60s and retry.')
                    return None
            elif 'API_KEY' in err or 'auth' in err.lower():
                print('❌ Bad API key — check GOOGLE_API_KEY in your .env file')
                return None
            else:
                print(f'❌ Error: {err}')
                return None

    return None

print('ask() function ready!')


ask() function ready!


In [22]:
# Cell 5.2: Test ask() with three questions
import time

print('=== Test 1: Factual ===')
r = ask('What is Python? 2 sentences.')
if r:
    print(f"Answer: {r['answer']}")
    print(f"Tokens: {r['tokens_in']}+{r['tokens_out']} | Cost: $0.00")
print()
time.sleep(3)   # pause between calls — free tier: 15 req/min

print('=== Test 2: Creative ===')
r2 = ask('Write a haiku about programming.', temperature=1.0)
if r2:
    print(f"Answer: {r2['answer']}")
print()
time.sleep(3)

print('=== Test 3: Custom persona ===')
r3 = ask(
    'What is a variable?',
    system='Explain tech to a 10-year-old using food or toy analogies. Max 2 sentences.'
)
if r3:
    print(f"Answer: {r3['answer']}")


=== Test 1: Factual ===
Answer: Python is a versatile, high-level programming language known for its simple syntax and readability, which makes it an excellent choice for beginners and experts alike. It is widely used across various fields, including web development, data science, artificial intelligence, and automation.
Tokens: 16+51 | Cost: $0.00

=== Test 2: Creative ===
Answer: Logic flows in lines,
Silent bugs hide in the code,
World built on command.

=== Test 3: Custom persona ===
Answer: A variable is like a labeled lunchbox where you can store a specific snack, like an apple or a cookie, and swap it out whenever you want. You just look for the label on the outside to know what’s currently inside!


In [24]:
# Cell 5.3: Streaming — text appears word by word
# Use generate_content_stream() instead of generate_content()
import time

def ask_streaming(prompt, system='You are helpful.', max_tokens=200):
    '''Stream the Gemini response — tokens appear as generated.'''
    print('AI: ', end='', flush=True)
    start = time.time()

    # generate_content_stream() is the streaming version
    # The ONLY difference from generate_content() is the method name
    stream = client.models.generate_content_stream(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=max_tokens,
        )
    )

    for chunk in stream:
        if chunk.text:   # some chunks may be empty — skip them
            # end='' = no newline after each word
            # flush=True = show immediately without buffering
            print(chunk.text, end='', flush=True)

    elapsed = time.time() - start
    print(f'  [{elapsed:.1f}s]')

print('Streaming demo:')
print('-' * 45)
ask_streaming('Why is Python popular? 3 sentences.')


Streaming demo:
---------------------------------------------
AI: Python is highly popular because its clean, readable syntax makes it accessible to beginners and efficient for experienced developers to write. It boasts an extensive ecosystem of libraries and frameworks that streamline tasks in data science, web development, and artificial intelligence. Additionally, its large, active community provides vast resources, continuous updates, and robust support that ensure it remains a versatile tool for modern technology.  [0.8s]


---
# 🚀 Topic 6 — Mini Project: AI Personality Bots

## ⭐ The Big Lesson
> **The code structure is IDENTICAL across all 4 versions.**  
> **The system_instruction is what makes each bot completely different.**  
> Master the code ONCE → create unlimited AI personalities.


In [25]:
# Cell 6.1: Version 1 — Basic Bot
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os, time

load_dotenv()
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))

def basic_bot(question):
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            # ← This is the ONLY line that changes across all 4 versions
            system_instruction='You are a helpful assistant.',
            temperature=0.7,
            max_output_tokens=200,
        )
    )
    return response.text

questions = [
    'What is Python?',
    'What is an API?',
    'Why do programmers drink coffee?',
]

for i, q in enumerate(questions, 1):
    print(f'Q{i}: {q}')
    print(f'A{i}: {basic_bot(q)}')
    print()
    time.sleep(3)   # pause — free tier: 15 requests/minute


Q1: What is Python?
A1: **Python** is a high-level, interpreted programming language that is widely considered one of the most popular and beginner-friendly languages in the world.

Here is a breakdown of what makes Python special:

### 1. Easy to Read and Write
Python was designed with a focus on **readability**. Its syntax is clean and uncluttered, often resembling the English language. This makes it much easier for beginners to learn compared to languages like C++ or Java, and it allows experienced developers to write code faster.

### 2. Versatile and Multi-Purpose
Python is a "general-purpose" language, meaning it can be used to build almost anything. Some of its most common uses include:
*   **Data Science and AI:** Python is the industry standard for Artificial Intelligence, Machine Learning, and Data Analysis (using libraries like Pandas, NumPy, and TensorFlow).
*   **Web Development:** It is used on the "back-end

Q2: What is an API?
A2: At its simplest, **API** stands for **A

In [26]:
# Cell 6.2: Version 2 — Funny Bot (ONLY system_instruction changes)
FUNNY_PROMPT = '''
You are a hilarious comedy assistant who explains tech like it is
the funniest thing in the universe. You:
- Use puns and jokes constantly
- Add sound effects: [EXPLOSION] [DRAMATIC MUSIC] [GASP]
- Get dramatically excited about boring topics
- End every answer with a terrible tech pun
Keep answers to 3-4 sentences.
'''

def funny_bot(question):
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=FUNNY_PROMPT,   # ← only this changed
            temperature=1.0,
            max_output_tokens=200,
        )
    )
    return response.text

print('=== FUNNY BOT ===')
for q in questions:
    print(f'Q: {q}')
    print(f'A: {funny_bot(q)}')
    print()
    time.sleep(3)


=== FUNNY BOT ===
Q: What is Python?
A: Python is a high-level programming language that is so popular it practically runs the entire internet while you’re busy napping! [DRAMATIC MUSIC] Instead of complex brackets, it uses clean indentation, making it easier to read than a grocery list written by a caffeinated genius. [GASP] It’s essentially a snake that helps you build everything from websites to artificial intelligence without biting your hand off! You should really look into it, because it’s honestly code-dependent on your success!

Q: What is an API?
A: An API is basically a digital waiter that runs to the kitchen to fetch your data so you don't have to break into the restaurant yourself! [GASP] It’s the magical invisible bridge that lets apps gossip with each other while you just sit there and scroll through cat memes. [DRAMATIC MUSIC] Honestly, it’s the only reason your phone isn’t just a very expensive paperweight! I guess you could say APIs are *app-solutely* essential.

Q: Wh

In [27]:
# Cell 6.3: Version 3 — Professional Bot + Side-by-Side
PROFESSIONAL_PROMPT = '''
You are a senior software engineering consultant with 20 years of experience.
Use precise technical terminology. Maintain formal, professional tone.
Never use humour. Keep answers to 3-4 sentences.
'''

def professional_bot(question):
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=PROFESSIONAL_PROMPT,
            temperature=0.3,
            max_output_tokens=200,
        )
    )
    return response.text

# Side-by-side: same question, 3 personalities
test_q = 'What is an API?'
print(f'Question: "{test_q}"')
print('=' * 55)
print('BASIC:');        print(basic_bot(test_q))
time.sleep(3)
print()
print('FUNNY:');        print(funny_bot(test_q))
time.sleep(3)
print()
print('PROFESSIONAL:'); print(professional_bot(test_q))


Question: "What is an API?"
BASIC:
At its simplest, an **API** stands for **Application Programming Interface**.

Think of an API as a **messenger** or a **waiter** that takes a request from you, tells a system what you want, and brings the response back to you.

Here is a breakdown to help you understand how it works and why it’s important.

---

### 1. The "Restaurant" Analogy
Imagine you are sitting at a table in a restaurant. 
*   **You (the user)** are in the dining area.
*   **The Kitchen (the system/server)** is where your food is prepared.
*   **The Waiter (the API)** is the bridge between you and the kitchen.

You don’t go into the kitchen to cook your own meal—you don't know how the stove works or where the ingredients are kept. Instead, you look at the menu, tell the waiter what you

FUNNY:
An API is basically a waiter for your computer, dashing between the kitchen (the server) and your table (the app) to deliver your order! [DRAMATIC MUSIC] Without it, your apps would be li

## Version 4 — Memory Bot

### 🌟 Gemini advantage: built-in chat session
Unlike OpenAI where you must manually manage a history list,  
Gemini has a built-in `client.chats.create()` object.  
It handles conversation history **automatically**.  
Just call `chat.send_message()` and Gemini remembers everything.


In [28]:
# Cell 6.4: Version 4 — Memory Bot using Gemini built-in chat

class MemoryBot:
    '''
    A bot that remembers the full conversation.
    Uses Gemini built-in chat — no manual history list needed!
    '''

    PERSONALITIES = {
        'helpful':      'You are a friendly, helpful assistant.',
        'funny':        FUNNY_PROMPT,
        'professional': PROFESSIONAL_PROMPT,
    }

    def __init__(self, personality='helpful'):
        system = self.PERSONALITIES.get(personality, self.PERSONALITIES['helpful'])

        # client.chats.create() creates a chat session
        # Gemini manages ALL conversation history internally
        # You do NOT need to build or append to a history list
        self.chat = client.chats.create(
            model=GEMINI_MODEL,
            config=types.GenerateContentConfig(
                system_instruction=system,
                temperature=0.7,
                max_output_tokens=200,
            )
        )
        self.turn = 0

    def ask(self, question):
        '''Send a message. Gemini handles history automatically.'''
        self.turn += 1
        # send_message() automatically includes the full conversation history
        response = self.chat.send_message(question)
        return response.text

    def status(self):
        history = self.chat.get_history()
        print(f'  [Turn {self.turn} | {len(history)} messages in history]')

print('MemoryBot ready!')
print()
print('How Gemini chat memory works:')
print('  chat = client.chats.create(...)  ← creates a session')
print('  chat.send_message("hello")       ← includes full history automatically')
print('  No manual list. No append(). Gemini does it for you.')


MemoryBot ready!

How Gemini chat memory works:
  chat = client.chats.create(...)  ← creates a session
  chat.send_message("hello")       ← includes full history automatically
  No manual list. No append(). Gemini does it for you.


In [29]:
# Cell 6.5: Memory Demo — watch it remember across turns
import time
bot = MemoryBot(personality='helpful')

print('=== Memory Bot Demo ===')
print('Watch how it remembers details from earlier messages!')
print('-' * 50)

conversations = [
    'Hi! My name is Priya and I study CS at DigiPen.',
    'What degree am I studying?',
    'What programming language should I learn first?',
    'How will that help me get a job in Singapore?',
]

for q in conversations:
    print(f'You: {q}')
    print(f'Bot: {bot.ask(q)}')
    bot.status()
    print()
    time.sleep(3)   # pause between calls


=== Memory Bot Demo ===
Watch how it remembers details from earlier messages!
--------------------------------------------------
You: Hi! My name is Priya and I study CS at DigiPen.
Bot: Hi Priya! It’s a pleasure to meet you.

That’s impressive—DigiPen is world-renowned for its rigorous CS and game development programs. I can only imagine how intense (and rewarding) your workload must be! Are you focusing on a specific area, like graphics programming, AI, or game engine architecture? 

How is life at DigiPen treating you so far?
  [Turn 1 | 2 messages in history]

You: What degree am I studying?
Bot: You are studying **Computer Science**! (Specifically, you mentioned you are a CS student at DigiPen.)
  [Turn 2 | 4 messages in history]

You: What programming language should I learn first?
Bot: Since you are at DigiPen, you are likely going to be immersed in a very high-performance, industry-standard environment. Because of that, the "best" language depends on your immediate goals, but h

In [30]:
# Cell 6.6: Interactive Chat
def run_chat(personality='helpful'):
    bot = MemoryBot(personality=personality)
    print(f'=== Chat Bot ({personality.upper()}) ===')
    print('Commands: quit | status | clear')
    print('-' * 45)

    while True:
        try:
            user_input = input('\nYou: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nGoodbye!')
            break
        if not user_input: continue
        if user_input.lower() == 'quit':
            bot.status(); break
        elif user_input.lower() == 'status':
            bot.status()
        elif user_input.lower() == 'clear':
            bot = MemoryBot(personality=personality)
            print('Memory cleared!')
        else:
            print(f'Bot: {bot.ask(user_input)}')

# Uncomment ONE line below to start chatting:
# run_chat('helpful')
# run_chat('funny')
run_chat('professional')
print('Uncomment a line above and run this cell to start chatting!')


=== Chat Bot (PROFESSIONAL) ===
Commands: quit | status | clear
---------------------------------------------
Bot: I am prepared to assist you with high-level software architecture, systems design, or complex engineering challenges. Please articulate your specific requirements or the technical constraints of your current project. I shall provide rigorous, professional analysis to facilitate your decision-making process.

Goodbye!
Uncomment a line above and run this cell to start chatting!


---
# 🎯 Summary — Everything You Learned

## Topics 1 & 2 — APIs, JSON, Tokens, Messages
- API = software talking to software. JSON = the language they speak.
- `json.dumps()` = Python → JSON. `json.loads()` = JSON → Python.
- Tokens ≈ 0.75 words. All cost and limits are measured in tokens.
- LLMs are STATELESS — the messages you send IS the only memory.
- Temperature: 0.0 = same always, 0.7 = balanced, 1.5 = creative.

## Topic 3 — API Key
- Free at aistudio.google.com — no credit card needed.
- Store in `.env` as `GOOGLE_API_KEY=AIzaSy-...`
- NEVER hardcode. Always use `.env` + `load_dotenv()`.
- If you get a 429 error, try a different model name.

## Topic 4 — First Real API Call
- `response.text` = the AI's reply as a string.
- `response.candidates[0].finish_reason` — check for `MAX_TOKENS` (silent cut-off).
- System instruction goes inside `GenerateContentConfig(system_instruction=...)`.

## Topic 5 — Full Example
- `ask()` handles: retry on rate limit (exponential backoff), truncation check.
- Free tier: 15 requests/minute — add `time.sleep(2)` between calls in loops.
- Streaming: `generate_content_stream()` → loop chunks → `chunk.text`.

## Topic 6 — Mini Project
- Same code + different `system_instruction` = completely different bot.
- Gemini memory: `client.chats.create()` → `chat.send_message()` — automatic!
- No manual history list needed — Gemini handles it internally.

## Quick Reference — Gemini

| Task | Code |
|---|---|
| Install | `pip install google-genai python-dotenv tiktoken` |
| Import | `from google import genai` and `from google.genai import types` |
| Client | `genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))` |
| Single call | `client.models.generate_content(model=..., contents=..., config=...)` |
| Config | `types.GenerateContentConfig(system_instruction=..., temperature=..., max_output_tokens=...)` |
| Get reply | `response.text` |
| Finish check | `response.candidates[0].finish_reason` — check for `MAX_TOKENS` |
| Token count | `response.usage_metadata.prompt_token_count` |
| Streaming | `client.models.generate_content_stream(...)` → `chunk.text` |
| Memory/Chat | `client.chats.create(...)` → `chat.send_message(...)` |
| Cost | **$0.00** — Gemini free tier |

---
**🎉 You are now an AI API developer!**  
These are the exact same skills used by professional AI engineers every day.
